# Tilt failure points: Kikuchi COM vs Friedel disk pair vs top/bottom halves

Failure point = first tilt, moving outwards from 0° on each side, at which the signal loses its
zero-tilt sign (linear interpolation between samples). Tolerance = the smaller of the two.

All three methods are evaluated on the **same** 4096-grid short-range simulations
(41 tilts, ±0.6°, 0.03° step), in the paper frame (`corrected_final`: P towards the top box,
`TILT_SIGN = -1` on the tilt axis), so the − / + sides match the figures.

- **Kikuchi COM**: corrected masked COM, read from `cache_shortrange/` (written by the short-range notebook).
- **Friedel disk pair**: every ROI ({100}, {200}, {300} × 1×1, 1×3, 1×5) computed here, best one picked
  per material by tolerance. Cached in `cache_failure/`.
- **Top/bottom halves**: whole-pattern intensity mismatch, also from `cache_shortrange/`.

In [1]:
import os, sys
import numpy as np

sys.path.insert(0, "..")
from kikuchi_com import mirror_box, stream_patterns, DISK_LATTICE, disk_row_box, reflection
from polarisation_direction import P_REVERSED

TILT_SIGN = -1        # paper x-axis convention, tilt axis only
MIN_SIGNAL = 1.0      # % ; ROIs with |value at 0| below this carry no usable sign
CACHE = "cache_failure"
os.makedirs(CACHE, exist_ok=True)

MATERIALS = {   # same files as the short-range / combined notebooks
    "PTO": dict(abbr="PTO", path="../data/short_range/PTO_short_range_21_sept_asymmetry_data.npz",
                orders=(1, 2, 3)),
    "BTO": dict(abbr="BTO", path="../data/short_range/BTO_kikuchi_final_15_slice_asymmetry_data.npz",
                orders=(1, 2, 3)),
    "Boracite": dict(abbr="Boracite",
                     path="../data/short_range/1_slice_thickness/boracite_kikuchi_1_slice_asymmetry_data.npz",
                     orders=(1, 2)),    # no {300} in boracite
}
WIDTHS = (1, 3, 5)

## Disk-pair asymmetry for every ROI (one streaming pass per 5 GB file)

In [2]:
def disk_rois(name, cfg, shape=(4096, 4096)):
    """{(order, n_disks): (box1, box2)} as ((y0, y1), (x0, x1)), half-open."""
    out = {}
    for o in cfg["orders"]:
        for w in WIDTHS:
            (lo, hi), xs = disk_row_box(shape, *DISK_LATTICE[name], o, w)
            a, b, _, _ = mirror_box((lo, hi, *xs), shape[0])
            out[(o, w)] = (((lo, hi), xs), ((a, b), xs))
    return out


def disk_asym(name, cfg):
    """(tilt, {roi: % asymmetry}), + = top (low row index) brighter. Cached."""
    f = os.path.join(CACHE, f"{name}_disks.npz")
    rois = disk_rois(name, cfg)
    keys = [f"{o}_{w}" for o, w in rois]
    if os.path.exists(f):
        d = np.load(f)
        if all(k in d.files for k in keys):
            return d["tilt"], {roi: d[k] for roi, k in zip(rois, keys)}
    sums = {roi: [] for roi in rois}
    for p in stream_patterns(cfg["path"]):
        for roi, (((y0, y1), (x0, x1)), ((Y0, Y1), _)) in rois.items():
            sums[roi].append((p[y0:y1, x0:x1].sum(), p[Y0:Y1, x0:x1].sum()))
    asym = {roi: (lambda s: (s[:, 0] - s[:, 1]) / (s[:, 0] + s[:, 1]) * 100)(np.array(v))
            for roi, v in sums.items()}
    tilt = np.load(cfg["path"], mmap_mode="r")["tilt_y_values"].astype(float)
    np.savez(f, tilt=tilt, **{k: asym[roi] for roi, k in zip(rois, keys)})
    return tilt, asym


DISKS = {name: disk_asym(name, cfg) for name, cfg in MATERIALS.items()}
for name, (t, a) in DISKS.items():
    print(name, len(t), "tilts,", len(a), "ROIs")

PTO 41 tilts, 9 ROIs
BTO 41 tilts, 9 ROIs
Boracite 41 tilts, 6 ROIs


## Kikuchi COM and top/bottom halves (short-range cache)

In [3]:
def kikuchi(name):
    d = np.load(f"cache_shortrange/{name}.npz")
    # caches hold raw values (+ towards higher row index); negate to + = top brighter
    return d["tilt"].astype(float), -d["com"], -d["mismatch"]


KIK = {name: kikuchi(name) for name in MATERIALS}
for name in MATERIALS:
    assert np.allclose(KIK[name][0], DISKS[name][0]), f"{name}: tilt grids differ"

## Paper frame and failure points

`final` frame: materials with P along −x (as simulated) are left alone, PTO is mirrored,
`y(t) → −y(−t)` (exact: mirroring the structure reverses both the tilt sense and the signal).
Then the tilt axis is multiplied by `TILT_SIGN`.

In [4]:
def to_paper(name, t, y):
    if not P_REVERSED[name]:                  # final: P towards the top box
        y = -y[::-1]                          # tilt grid is symmetric + ascending
    assert np.allclose(t, -t[::-1])
    t = TILT_SIGN * t
    order = np.argsort(t)
    return t[order], y[order]


def failure_points(t, y):
    """(t_minus, t_plus): first loss of the zero-tilt sign on each side, interpolated.
    None = sign kept over the whole sampled range on that side."""
    i0 = int(np.argmin(np.abs(t)))
    s0 = np.sign(y[i0])
    out = []
    for step in (-1, +1):
        tc = None
        i = i0
        while 0 <= i + step < len(t):
            j = i + step
            if np.sign(y[j]) != s0:
                tc = t[i] - y[i] * (t[j] - t[i]) / (y[j] - y[i])
                break
            i = j
        out.append(tc)
    return tuple(out)


# self-check on a synthetic curve: sign flips at -0.1 and +0.25
_t = np.linspace(-0.6, 0.6, 41)
_fp = failure_points(_t, (_t + 0.1) * (0.25 - _t))
assert np.allclose(_fp, (-0.1, 0.25), atol=2e-3), _fp   # linear interp of a parabola
assert failure_points(_t, np.ones_like(_t)) == (None, None)

In [5]:
rows = []   # (method, material, roi label, value at 0, unit, t-, t+, tol, t_max)
for name, cfg in MATERIALS.items():
    t_raw, com, mism = KIK[name]
    for method, y, roi, unit in (("Kikuchi COM", com, "full box", "px"),
                                 ("Top/bottom halves", mism, "whole pattern", "%")):
        t, yy = to_paper(name, t_raw, y)
        rows.append(dict(method=method, material=cfg["abbr"], roi=roi,
                         y0=yy[np.argmin(np.abs(t))], unit=unit, fp=failure_points(t, yy),
                         t_max=np.abs(t).max()))
    t_raw, asym = DISKS[name]
    for (o, w), y in asym.items():
        t, yy = to_paper(name, t_raw, y)
        rows.append(dict(method="Disk pair (all)", material=cfg["abbr"],
                         roi=f"{{{o}00}} 1x{w}",
                         y0=yy[np.argmin(np.abs(t))], unit="%", fp=failure_points(t, yy),
                         t_max=np.abs(t).max()))

for r in rows:
    fp = r["fp"]
    r["tol"] = min(abs(v) if v is not None else r["t_max"] for v in fp)
    r["tol_open"] = all(v is None for v in fp)      # sign kept over the full range


def fmt(v):
    return "  none " if v is None else f"{v:+.3f}"


print(f"{'method':18s} {'mat':9s} {'ROI':14s} {'value@0':>9s}   {'fail -':>7s} {'fail +':>7s}  tol")
for r in rows:
    print(f"{r['method']:18s} {r['material']:9s} {r['roi']:14s} {r['y0']:+8.2f}{r['unit']:<2s}"
          f" {fmt(r['fp'][0]):>7s} {fmt(r['fp'][1]):>7s}  {r['tol']:.3f}")

method             mat       ROI              value@0    fail -  fail +  tol
Kikuchi COM        PTO       full box         +24.25px  -0.090  +0.311  0.090
Top/bottom halves  PTO       whole pattern     -7.73%    none   +0.055  0.055
Disk pair (all)    PTO       {100} 1x1         +9.44%   -0.008  +0.243  0.008
Disk pair (all)    PTO       {100} 1x3        -31.18%    none   +0.140  0.140
Disk pair (all)    PTO       {100} 1x5        -30.33%    none   +0.135  0.135
Disk pair (all)    PTO       {200} 1x1        -74.48%    none   +0.042  0.042
Disk pair (all)    PTO       {200} 1x3        -60.15%    none   +0.036  0.036
Disk pair (all)    PTO       {200} 1x5        -60.26%    none   +0.040  0.040
Disk pair (all)    PTO       {300} 1x1        -33.56%    none   +0.025  0.025
Disk pair (all)    PTO       {300} 1x3         -8.47%   -0.020  +0.045  0.020
Disk pair (all)    PTO       {300} 1x5         -8.39%    none   +0.037  0.037
Kikuchi COM        BTO       full box          +8.17px  -0.015   

## Best Friedel disk pair per material

The table uses the ROIs shown in the paper figures (`FIGURE_ROI`), so it matches them. The ROI with the
largest tolerance (among those with |value at 0°| ≥ `MIN_SIGNAL`; boracite {200} 1×3 is ≈ 0 at zero tilt)
is printed alongside, so any mismatch is visible.

In [6]:
FIGURE_ROI = {"PTO": "{100} 1x3", "BTO": "{100} 1x3", "Boracite": "{100} 1x1"}   # combined / less-disks figs

best = {}
for name, cfg in MATERIALS.items():
    m = cfg["abbr"]
    cands = [r for r in rows if r["method"] == "Disk pair (all)" and r["material"] == m]
    best[m] = next(r for r in cands if r["roi"] == FIGURE_ROI[m])
    top = max((r for r in cands if abs(r["y0"]) >= MIN_SIGNAL), key=lambda r: (round(r["tol"], 4), abs(r["y0"])))
    for tag, r in (("figure", best[m]), ("largest tol", top)):
        print(f"{m:9s} {tag:12s} {r['roi']:10s} {r['y0']:+6.1f} %  fail {fmt(r['fp'][0])} / "
              f"{fmt(r['fp'][1])}  tol {r['tol']:.3f} deg")

PTO       figure       {100} 1x3   -31.2 %  fail   none  / +0.140  tol 0.140 deg
PTO       largest tol  {100} 1x3   -31.2 %  fail   none  / +0.140  tol 0.140 deg
BTO       figure       {100} 1x3   -39.3 %  fail -0.042 / +0.114  tol 0.042 deg
BTO       largest tol  {100} 1x5   -35.8 %  fail -0.052 / +0.107  tol 0.052 deg
Boracite  figure       {100} 1x1   +68.6 %  fail -0.460 / +0.176  tol 0.176 deg
Boracite  largest tol  {100} 1x1   +68.6 %  fail -0.460 / +0.176  tol 0.176 deg


## Summary table and LaTeX

In [7]:
METHODS = ["Kikuchi COM", "Friedel disk pair", "Top/bottom halves"]
summary = []
for method in METHODS:
    for m in (c["abbr"] for c in MATERIALS.values()):
        if method == "Friedel disk pair":
            r = best[m]
        else:
            r = next(r for r in rows if r["method"] == method and r["material"] == m)
        summary.append(dict(r, method=method))

for r in summary:
    print(f"{r['method']:18s} {r['material']:9s} {r['roi']:14s} {r['y0']:+8.2f} {r['unit']:2s}"
          f" {fmt(r['fp'][0]):>7s} {fmt(r['fp'][1]):>7s}  tol {r['tol']:.3f}"
          f"{' (sign kept over full range)' if r['tol_open'] else ''}")

Kikuchi COM        PTO       full box         +24.25 px  -0.090  +0.311  tol 0.090
Kikuchi COM        BTO       full box          +8.17 px  -0.015   none   tol 0.015
Kikuchi COM        Boracite  full box         +12.48 px  -0.020   none   tol 0.020
Friedel disk pair  PTO       {100} 1x3        -31.18 %    none   +0.140  tol 0.140
Friedel disk pair  BTO       {100} 1x3        -39.27 %   -0.042  +0.114  tol 0.042
Friedel disk pair  Boracite  {100} 1x1        +68.63 %   -0.460  +0.176  tol 0.176
Top/bottom halves  PTO       whole pattern     -7.73 %    none   +0.055  tol 0.055
Top/bottom halves  BTO       whole pattern     -4.73 %    none   +0.022  tol 0.022
Top/bottom halves  Boracite  whole pattern     +4.89 %   -0.011   none   tol 0.011


In [8]:
KPP = {"PTO": 1 / (30 * 3.9423), "BTO": 1 / (30 * 4.0230), "Boracite": 1 / (7 * 17.1032 * np.sqrt(2))}
# one-column layout (revtex twocolumn, ~8.6 cm): units in the group headers, degrees in the
# column headers, disk ROI folded into the material cell
GROUPS = {"Kikuchi COM": r"Kikuchi COM (\AA$^{-1}$)", "Friedel disk pair": r"Friedel disk pair (\%)",
          "Top/bottom halves": r"Top/bottom halves (\%)"}
NUM = lambda v, d: f"${v:+.{d}f}$"


def material(r):
    if r["method"] != "Friedel disk pair":
        return r["material"]
    o, w = r["roi"].split()
    return r["material"] + ", " + reflection(r["material"], o[1]) + " " + w.replace("x", r"$\times$")


def side(v, t_max, sgn):
    return (rf"$<-{t_max:.1f}$" if sgn < 0 else rf"$>{t_max:.1f}$") if v is None else NUM(v, 3)


lines = [r"\begin{table}[htbp]",
         r"\caption{Tilt tolerance of the three polarisation metrics, evaluated on the same simulated "
         r"patterns (tilt about $y$, along the polar axis; 0.03$^\circ$ steps over $\pm0.6^\circ$). "
         r"$S_0$: signal at zero tilt. $t_-$, $t_+$: first tilt on each side of zero at which the "
         r"signal loses its zero-tilt sign (linear interpolation); $\theta_\mathrm{tol}$ is the smaller "
         r"of the two. $>0.6$ means the sign is kept over the whole simulated range on that side. "
         r"For the Friedel disk pair method the best-performing ROI is given.}",
         r"\label{tab:tilt-failure}",
         r"\small",
         r"\setlength{\tabcolsep}{4pt}",
         r"\begin{tabular}{lrrrr}",
         r"\toprule",
         r" & $S_0$ & $t_-$ ($^\circ$) & $t_+$ ($^\circ$) & $\theta_\mathrm{tol}$ ($^\circ$) \\",
         r"\midrule"]
for k, method in enumerate(METHODS):
    if k:
        lines.append(r"\addlinespace")
    lines.append(rf"\multicolumn{{5}}{{l}}{{\textit{{{GROUPS[method]}}}}} \\")
    for r in (r for r in summary if r["method"] == method):
        s0 = NUM(r["y0"] * KPP[r["material"]], 3) if method == "Kikuchi COM" else NUM(r["y0"], 1)
        tol = rf"$>{r['t_max']:.1f}$" if r["tol_open"] else f"${r['tol']:.3f}$"
        lines.append(rf"{material(r)} & {s0} & {side(r['fp'][0], r['t_max'], -1)} & "
                     rf"{side(r['fp'][1], r['t_max'], +1)} & {tol} \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
LATEX = "\n".join(lines)

os.makedirs("figures/corrected_final", exist_ok=True)
with open("figures/corrected_final/tilt_failure_table.tex", "w") as f:
    f.write(LATEX + "\n")
print("% needs \\usepackage{booktabs}\n")
print(LATEX)

% needs \usepackage{booktabs}

\begin{table}[htbp]
\caption{Tilt tolerance of the three polarisation metrics, evaluated on the same simulated patterns (tilt about $y$, along the polar axis; 0.03$^\circ$ steps over $\pm0.6^\circ$). $S_0$: signal at zero tilt. $t_-$, $t_+$: first tilt on each side of zero at which the signal loses its zero-tilt sign (linear interpolation); $\theta_\mathrm{tol}$ is the smaller of the two. $>0.6$ means the sign is kept over the whole simulated range on that side. For the Friedel disk pair method the best-performing ROI is given.}
\label{tab:tilt-failure}
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrr}
\toprule
 & $S_0$ & $t_-$ ($^\circ$) & $t_+$ ($^\circ$) & $\theta_\mathrm{tol}$ ($^\circ$) \\
\midrule
\multicolumn{5}{l}{\textit{Kikuchi COM (\AA$^{-1}$)}} \\
PTO & $+0.205$ & $-0.090$ & $+0.311$ & $0.090$ \\
BTO & $+0.068$ & $-0.015$ & $>0.6$ & $0.015$ \\
Boracite & $+0.074$ & $-0.020$ & $>0.6$ & $0.020$ \\
\addlinespace
\multicolumn{5}{l}{\textit